# Most recent benchmarks for BRCA dataset

In [26]:
%load_ext autoreload
%autoreload 2

import numpy as np
import polars as pl
import torch
import torch_geometric as pyg
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold

from bipartite_gnn.graph_visualizatons import visualize_graph, visualize_embeddings
from baseline_evals.feature_selection import variance_filtering, class_variational_selection
from bipartite_gnn.preprocessing import mrmr_selection
from baseline_evals.knn_eval import knn_eval
from baseline_evals.svm_eval import svm_eval
from baseline_evals.xgboost_eval import xgboost_eval
from baseline_evals.mlp_eval import mlp_eval

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load data and make sure that everything is aligned

In [3]:
null_vals = ["NA"]

mrna = pl.read_csv("BRCA_PROCESSED_DATA/mrna.tsv", separator="\t", null_values=null_vals)
meth = pl.read_csv("BRCA_PROCESSED_DATA/meth.tsv", separator="\t", null_values=null_vals)
mirna = pl.read_csv("BRCA_PROCESSED_DATA/mirna.tsv", separator="\t", null_values=null_vals)
cnv = pl.read_csv("BRCA_PROCESSED_DATA/cnvth.tsv", separator="\t", null_values=null_vals)

labels = pl.read_csv("BRCA_PROCESSED_DATA/labels.tsv", separator="\t")
le = LabelEncoder()
le.fit(labels["PAM50_mRNA_nature2012"].to_list())
y = le.transform(labels["PAM50_mRNA_nature2012"].to_list())

assert mrna.columns[1:] == meth.columns[1:] == cnv.columns[1:] == mirna.columns[1:] == labels["sampleID"].to_list()

In [6]:
# concat all data into one dataframe
mrna_X = mrna[:, 1:].to_numpy().T
meth_X = meth[:, 1:].to_numpy().T
mirna_X = mirna[:, 1:].to_numpy().T
cnv_X = cnv[:, 1:].to_numpy().T

# get 5000 most variant features in mrna_X, meth_X and cnv_X
# mrna_indices = variance_filtering(mrna_X, 1000)
# meth_indices = variance_filtering(meth_X, 1000)
# cnv_indices = variance_filtering(cnv_X, 1000)

# mrna_X = mrna_X[:, mrna_indices]
# meth_X = meth_X[:, meth_indices]
# cnv_X = cnv_X[:, cnv_indices]

print(mrna_X.shape, meth_X.shape, mirna_X.shape, cnv_X.shape)

# X_list = [mrna_X, meth_X, cnv_X, mirna_X]

X = np.hstack([mrna_X, cnv_X, mirna_X])

print(X.shape)

(483, 18975) (483, 9317) (483, 231) (483, 24776)
(483, 43982)


In [15]:
knn_eval(X, y, select_n_features=True)

500
| KNN | 0.84 +/- 0.02 | 0.82 +/- 0.03 | 0.83 +/- 0.02 |
study.best_value=0.8298658475554511, study.best_params={'n_neighbors': 9, 'n_features': 3686}


| KNN | 0.81 +/- 0.03 | 0.79 +/- 0.03 | 0.79 +/- 0.04 |

In [35]:
svm_eval(X, y, select_n_features=True, n_trials=20, n_features_preselect=10000)

Trial 1 / 20
| SVM | 0.81 +/- 0.03 | 0.81 +/- 0.02 | 0.82 +/- 0.02 |
Trial 2 / 20
| SVM | 0.82 +/- 0.01 | 0.82 +/- 0.01 | 0.83 +/- 0.01 |
Trial 3 / 20
| SVM | 0.83 +/- 0.01 | 0.83 +/- 0.01 | 0.83 +/- 0.01 |
Trial 4 / 20
| SVM | 0.83 +/- 0.01 | 0.83 +/- 0.01 | 0.83 +/- 0.01 |
Trial 5 / 20
Trial 6 / 20
Pruning trial
Trial 7 / 20
| SVM | 0.83 +/- 0.02 | 0.83 +/- 0.02 | 0.83 +/- 0.02 |
Trial 8 / 20
Trial 9 / 20
Pruning trial
Trial 10 / 20
Trial 11 / 20
Trial 12 / 20
| SVM | 0.83 +/- 0.01 | 0.83 +/- 0.02 | 0.83 +/- 0.01 |
Trial 13 / 20
Pruning trial
Trial 14 / 20
Pruning trial
Trial 15 / 20
Pruning trial
Trial 16 / 20
Pruning trial
Trial 17 / 20
Pruning trial
Trial 18 / 20
| SVM | 0.83 +/- 0.01 | 0.83 +/- 0.02 | 0.83 +/- 0.01 |
Trial 19 / 20
Pruning trial
Trial 20 / 20
| LIN SVM | 0.83 +/- 0.01 | 0.83 +/- 0.02 | 0.83 +/- 0.01 |
study.best_value=0.8340112030694543, study.best_params={'C': 0.06063679285300764, 'class_weight': 'balanced', 'n_features': 999}


{'acc': 0.832323883161512,
 'f1_macro': 0.8289296230999007,
 'f1_weighted': 0.8340112030694543,
 'acc_std': 0.013420635424690802,
 'f1_macro_std': 0.021639036998337795,
 'f1_weighted_std': 0.011916820989229982}

In [ ]:
| LIN SVM | 0.83 +/- 0.01 | 0.83 +/- 0.02 | 0.83 +/- 0.01 |

In [37]:
xgboost_eval(X, y, n_trials=20, n_features=10000)

0 / 20
ACC: [0.82474227 0.73195876 0.68041237 0.76041667 0.78125   ]
F1M: [0.77534965 0.65301694 0.65819658 0.7557673  0.77063223]
F1W: [0.82192264 0.68485547 0.68005714 0.75984802 0.78102066]
| XGBoost | 0.76 +/- 0.05 | 0.72 +/- 0.06 | 0.75 +/- 0.06 |
1 / 20
ACC: [0.78350515 0.81443299 0.80412371 0.85416667 0.82291667]
F1M: [0.74478198 0.82475369 0.79336456 0.85538694 0.81930147]
F1W: [0.78254686 0.8151152  0.7979026  0.84959848 0.82368004]
| XGBoost | 0.82 +/- 0.02 | 0.81 +/- 0.04 | 0.81 +/- 0.02 |
2 / 20
ACC: [0.79381443 0.81443299 0.79381443 0.8125     0.83333333]
F1M: [0.75073128 0.8259487  0.78273514 0.80533597 0.82766712]
F1W: [0.79034007 0.81435883 0.78697587 0.80970026 0.83414654]
3 / 20
Pruning trial
ACC: [0.73195876 0.7628866  0.71134021 0.78125    0.        ]
F1M: [0.68371131 0.70753077 0.67854313 0.77198625 0.        ]
F1W: [0.73725572 0.76093947 0.7017254  0.78257003 0.        ]
4 / 20
ACC: [0.8556701  0.86597938 0.88659794 0.86458333 0.82291667]
F1M: [0.82685185 0.874692

| XGBoost | 0.88 +/- 0.03 | 0.87 +/- 0.04 | 0.88 +/- 0.03 |

In [38]:
mlp_eval(X, y, reg_type=None, n_trials=25, n_features_preselect=10000)

Trial 0 / 25
Eval 1 / 5


/home/lubojjan/micromamba/envs/diploma/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:75: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default


Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.75510204 0.87755102 0.71428573 0.83333331 0.72916669]
[0.72379863 0.86841238 0.70272109 0.79725877 0.72252415]
[0.74614487 0.87375978 0.70109677 0.82311769 0.73294082]
{'acc': 0.7818877577781678, 'f1_macro': 0.7629430053347183, 'f1_weighted': 0.7754119844824131, 'acc_std': 0.06303193616349628, 'f1_macro_std': 0.06179645127331436, 'f1_weighted_std': 0.06345272118442748}
Trial 1 / 25
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.81632656 0.81632656 0.87755102 0.89583331 0.77083331]
[0.80772727 0.82140523 0.88312648 0.91388045 0.77475158]
[0.81721707 0.81731359 0.87457181 0.89171733 0.77207543]
{'acc': 0.8353741526603699, 'f1_macro': 0.840178201134259, 'f1_weighted': 0.834579045921231, 'acc_std': 0.04544301882443778, 'f1_macro_std': 0.05091700148803954, 'f1_weighted_std': 0.043290739822217164}
Trial 2 / 25
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.81801474 0.90808821 0.89430147 0.90625    0.765625  ]
[0.82099724 0.

Exception ignored in: <function _releaseLock at 0x73dada78cea0>
Traceback (most recent call last):
  File "/home/lubojjan/micromamba/envs/diploma/lib/python3.12/logging/__init__.py", line 243, in _releaseLock
    def _releaseLock():
    
KeyboardInterrupt: 


Eval 5 / 5
[0.83673471 0.81632656 0.87755102 0.89583331 0.77083331]
[0.78084416 0.81548432 0.89183502 0.89671202 0.77475158]
[0.82798834 0.81286441 0.87830688 0.89165722 0.77207543]
Trial 6 / 25
Eval 1 / 5


Exception ignored in: <function _releaseLock at 0x73dada78cea0>
Traceback (most recent call last):
  File "/home/lubojjan/micromamba/envs/diploma/lib/python3.12/logging/__init__.py", line 243, in _releaseLock
    def _releaseLock():
    
KeyboardInterrupt: 


| MLP no reg | 0.86 +/- 0.06 | 0.83 +/- 0.09 | 0.86 +/- 0.05 |

In [4]:
mlp_eval(X, y, reg_type="l1", n_trials=25, n_features_preselect=10000)

Trial 0 / 25
Eval 1 / 5


/home/lubojjan/micromamba/envs/diploma/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:75: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default


Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.85714287 0.85714287 0.85714287 0.875      0.77083331]
[0.82492754 0.86745815 0.87248818 0.8488319  0.77475158]
[0.85157054 0.85827691 0.8554663  0.8673212  0.77207543]
{'acc': 0.8434523820877076, 'f1_macro': 0.8376914694398387, 'f1_weighted': 0.8409420765482952, 'acc_std': 0.03696233040462072, 'f1_macro_std': 0.035623933126147554, 'f1_weighted_std': 0.0348220827526417}
Trial 1 / 25
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.83673471 0.89795917 0.87755102 0.85416669 0.79166669]
[0.81616833 0.90137803 0.88958103 0.85741314 0.81045344]
[0.8370059  0.89957102 0.87960389 0.84869854 0.78952298]
{'acc': 0.8516156554222107, 'f1_macro': 0.8549987927768907, 'f1_weighted': 0.8508804653947226, 'acc_std': 0.0364477538231228, 'f1_macro_std': 0.03699969677222398, 'f1_weighted_std': 0.03783599010559446}
Trial 2 / 25
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.83673471 0.85714287 0.73469388 0.875      0.75      ]
[0.73333333 0.

Exception ignored in: <function _releaseLock at 0x7a170c334ea0>
Traceback (most recent call last):
  File "/home/lubojjan/micromamba/envs/diploma/lib/python3.12/logging/__init__.py", line 243, in _releaseLock
    def _releaseLock():
    
KeyboardInterrupt: 


| MLP l1 reg | 0.87 +/- 0.06 | 0.85 +/- 0.09 | 0.87 +/- 0.05 |

{'acc': 0.867463231086731, 'f1_macro': 0.8446657147200625, 'f1_weighted': 0.8687115181280782, 'acc_std': 0.0655361141104709, 'f1_macro_std': 0.09939143723563133, 'f1_weighted_std': 0.05932617979442903}

In [4]:
mlp_eval(X, y, reg_type="inner_mat", n_trials=25, n_features_preselect=10000)

Trial 0 / 25
Eval 1 / 5


/home/lubojjan/micromamba/envs/diploma/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:75: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default


Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.75510204 0.89795917 0.83673471 0.83333331 0.77083331]
[0.76410799 0.90137803 0.8544686  0.86561265 0.75076628]
[0.75960252 0.89957102 0.83308686 0.82938076 0.77159962]
{'acc': 0.8187925100326539, 'f1_macro': 0.8272667099098514, 'f1_weighted': 0.8186481559786781, 'acc_std': 0.05129771686262821, 'f1_macro_std': 0.05923543169406324, 'f1_weighted_std': 0.050144044882010554}
Trial 1 / 25
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.81632656 0.85714287 0.87755102 0.9375     0.79166669]
[0.78222222 0.86313503 0.88312648 0.94832827 0.78913309]
[0.81297052 0.85515115 0.87457181 0.9350304  0.79250611]
{'acc': 0.8560374259948731, 'f1_macro': 0.8531890166223708, 'f1_weighted': 0.8540459975191783, 'acc_std': 0.05063744215757385, 'f1_macro_std': 0.06194529762727774, 'f1_weighted_std': 0.049910632703314174}
Trial 2 / 25
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.87755102 0.81632656 0.75510204 0.89583331 0.79166669]
[0.84230769

| MLP inner prod | 0.86 +/- 0.05 | 0.85 +/- 0.06 | 0.85 +/- 0.05 |

{'acc': 0.8560374259948731, 'f1_macro': 0.8531890166223708, 'f1_weighted': 0.8540459975191783, 'acc_std': 0.05063744215757385, 'f1_macro_std': 0.06194529762727774, 'f1_weighted_std': 0.049910632703314174}

# Results

| Method | ACC | F1 | F1 Weighted |
| --- | --- | --- | --- |
| KNN | 0.80 +/- 0.04 | 0.79 +/- 0.04 | 0.79 +/- 0.04 |
| LIN SVM w RFE | 0.84 +/- 0.01 | 0.84 +/- 0.02 | 0.84 +/- 0.01 |
| XGBoost | 0.88 +/- 0.02 | 0.88 +/- 0.03 | 0.88 +/- 0.02 |
| MLP no reg | 0.88 +/- 0.07 | 0.88 +/- 0.06 | 0.88 +/- 0.07 |
| MLP l1 reg | 0.87 +/- 0.06 | 0.85 +/- 0.09 | 0.87 +/- 0.05 |
| MLP inner prod | 0.86 +/- 0.05 | 0.85 +/- 0.06 | 0.85 +/- 0.05 |
| MOGONET (mrna, cnv, mirna) | 0.766 ± 0.025 | 0.742 ± 0.027| 0.768 ± 0.024|
| MOGONET (mrna, meth, mirna) | 0.824 ± 0.029 | 0.812 ± 0.039| 0.824 ± 0.032|

In [30]:
# prepare 5 folds for cross validation
skf = StratifiedKFold(n_splits=5)

In [31]:
train_indices = []
test_indices = []

for train_index, test_index in skf.split(X, y):
    train_indices.append(train_index)
    test_indices.append(test_index)

train_idx = train_indices[0]
test_idx = test_indices[0]

In [36]:
mrna_gene_names = mrna[:, 0].to_numpy()
meth_gene_names = meth[:, 0].to_numpy()
cna_gene_names = cnv[:, 0].to_numpy()
mirna_gene_names = mirna[:, 0].to_numpy()

mrna_X_train = mrna_X[train_idx]
meth_X_train = meth_X[train_idx]
cnv_X_train = cnv_X[train_idx]
mirna_X_train = mirna_X[train_idx]

y_train = y[train_idx]

mrna_X_test = mrna_X[test_idx]
meth_X_test = meth_X[test_idx]
cnv_X_test = cnv_X[test_idx]
mirna_X_test = mirna_X[test_idx]

y_test = y[test_idx]

mrna_mask = class_variational_selection(mrna_X_train, y_train, n_features=500)
meth_mask = class_variational_selection(meth_X_train, y_train, n_features=500)
cnv_mask = class_variational_selection(cnv_X_train, y_train, n_features=500)
mirna_mask = class_variational_selection(mirna_X_train, y_train, n_features=200)

mrna_X_train = mrna_X_train[:, mrna_mask]
meth_X_train = meth_X_train[:, meth_mask]
cnv_X_train = cnv_X_train[:, cnv_mask]
mirna_X_train = mirna_X_train[:, mirna_mask]

mrna_X_test = mrna_X_test[:, mrna_mask]
meth_X_test = meth_X_test[:, meth_mask]
cnv_X_test = cnv_X_test[:, cnv_mask]
mirna_X_test = mirna_X_test[:, mirna_mask]

mrna_gene_names = mrna_gene_names[mrna_mask]
meth_gene_names = meth_gene_names[meth_mask]
cna_gene_names = cna_gene_names[cnv_mask]
mirna_gene_names = mirna_gene_names[mirna_mask]